# 04 Modeling PySpark ML

Notebook ini digunakan untuk membangun model machine learning berbasis PySpark ML untuk klasifikasi tingkat permintaan produk e-commerce Indonesia.

Target klasifikasi yang digunakan adalah `demand_level`, dengan tiga kelas:

1. Low Demand
2. Medium Demand
3. High Demand

Model yang digunakan:

1. Logistic Regression
2. Decision Tree
3. Random Forest

Tahapan utama notebook ini meliputi membaca dataset hasil preprocessing, menyiapkan fitur numerik dan kategorikal, melakukan encoding fitur kategorikal, membagi data train-test, melatih model, mengevaluasi performa model, dan menentukan model terbaik berdasarkan F1-score.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Inisialisasi Direktori Project

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATASET_DIR = BASE_DIR / "datasets"
PROCESSED_DIR = DATASET_DIR / "processed"

RESULTS_DIR = BASE_DIR / "results"
MODELING_DIR = RESULTS_DIR / "modeling"

MODELING_DIR.mkdir(parents=True, exist_ok=True)

processed_csv_path = PROCESSED_DIR / "ecommerce_demand_processed.csv"

print("Base directory      :", BASE_DIR)
print("Processed dataset   :", processed_csv_path)
print("Modeling result dir :", MODELING_DIR)

Base directory      : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code
Processed dataset   : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\processed\ecommerce_demand_processed.csv
Modeling result dir : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling


## Inisialisasi Spark Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("E-Commerce Demand Classification")
    .getOrCreate()
)

spark

## Membaca Dataset Hasil Preprocessing

In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(processed_csv_path))
)

print("Jumlah baris:", df.count())
print("Jumlah kolom:", len(df.columns))

df.printSchema()

Jumlah baris: 17416
Jumlah kolom: 21
root
 |-- total_weight_gr: integer (nullable = true)
 |-- Total Diskon: integer (nullable = true)
 |-- num_product_categories: integer (nullable = true)
 |-- Ongkos Kirim Dibayar oleh Pembeli: integer (nullable = true)
 |-- Estimasi Potongan Biaya Pengiriman: integer (nullable = true)
 |-- Total Pembayaran: integer (nullable = true)
 |-- Perkiraan Ongkos Kirim: integer (nullable = true)
 |-- discount_ratio: double (nullable = true)
 |-- shipping_ratio: double (nullable = true)
 |-- is_cod: integer (nullable = true)
 |-- has_shipping_discount: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- product_categories: string (nullable = true)
 |-- Opsi Pengiriman: string (nullable = true)
 |-- Metode Pembayaran: string (nullable = true)
 |-- Kota/Kabupaten: string (nullable = true)
 |-- Provinsi: string (nullable = true)
 |-- net_qty: integer (nullable = true)
 |-- demand_level: string (nu

In [5]:
df.show(5, truncate=False)

+---------------+------------+----------------------+---------------------------------+----------------------------------+----------------+----------------------+--------------+-------------------+------+---------------------+----------+-----------+-----------------------+-------------------------------+---------------------+--------------+----------+-------+-------------+-----+
|total_weight_gr|Total Diskon|num_product_categories|Ongkos Kirim Dibayar oleh Pembeli|Estimasi Potongan Biaya Pengiriman|Total Pembayaran|Perkiraan Ongkos Kirim|discount_ratio|shipping_ratio     |is_cod|has_shipping_discount|order_year|order_month|product_categories     |Opsi Pengiriman                |Metode Pembayaran    |Kota/Kabupaten|Provinsi  |net_qty|demand_level |label|
+---------------+------------+----------------------+---------------------------------+----------------------------------+----------------+----------------------+--------------+-------------------+------+---------------------+----------

## Mengecek Distribusi Target

In [6]:
(
    df
    .groupBy("demand_level", "label")
    .count()
    .orderBy("label")
    .show(truncate=False)
)

+-------------+-----+-----+
|demand_level |label|count|
+-------------+-----+-----+
|Low Demand   |0    |10288|
|Medium Demand|1    |5027 |
|High Demand  |2    |2101 |
+-------------+-----+-----+



## Menentukan Fitur Numerik dan Kategorikal

In [7]:
numeric_features = [
    "total_weight_gr",
    "Total Diskon",
    "num_product_categories",
    "Ongkos Kirim Dibayar oleh Pembeli",
    "Estimasi Potongan Biaya Pengiriman",
    "Total Pembayaran",
    "Perkiraan Ongkos Kirim",
    "discount_ratio",
    "shipping_ratio",
    "is_cod",
    "has_shipping_discount",
    "order_year",
    "order_month"
]

categorical_features = [
    "product_categories",
    "Opsi Pengiriman",
    "Metode Pembayaran",
    "Kota/Kabupaten",
    "Provinsi"
]

numeric_features = [feature for feature in numeric_features if feature in df.columns]
categorical_features = [feature for feature in categorical_features if feature in df.columns]

print("Fitur numerik:")
for feature in numeric_features:
    print("-", feature)

print("Fitur kategorikal:")
for feature in categorical_features:
    print("-", feature)

Fitur numerik:
- total_weight_gr
- Total Diskon
- num_product_categories
- Ongkos Kirim Dibayar oleh Pembeli
- Estimasi Potongan Biaya Pengiriman
- Total Pembayaran
- Perkiraan Ongkos Kirim
- discount_ratio
- shipping_ratio
- is_cod
- has_shipping_discount
- order_year
- order_month
Fitur kategorikal:
- product_categories
- Opsi Pengiriman
- Metode Pembayaran
- Kota/Kabupaten
- Provinsi


In [8]:
df = df.withColumn("label", col("label").cast("double"))

df.select("demand_level", "label").show(5)

+-------------+-----+
| demand_level|label|
+-------------+-----+
|Medium Demand|  1.0|
|Medium Demand|  1.0|
|   Low Demand|  0.0|
|   Low Demand|  0.0|
|   Low Demand|  0.0|
+-------------+-----+
only showing top 5 rows


## Membagi Data Train dan Test

Data dibagi menjadi 80% data latih dan 20% data uji.

In [9]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Jumlah data train:", train_df.count())
print("Jumlah data test :", test_df.count())

Jumlah data train: 13990
Jumlah data test : 3426


In [10]:
print("Distribusi label pada data train:")
(
    train_df
    .groupBy("demand_level", "label")
    .count()
    .orderBy("label")
    .show(truncate=False)
)

print("Distribusi label pada data test:")
(
    test_df
    .groupBy("demand_level", "label")
    .count()
    .orderBy("label")
    .show(truncate=False)
)

Distribusi label pada data train:
+-------------+-----+-----+
|demand_level |label|count|
+-------------+-----+-----+
|Low Demand   |0.0  |8314 |
|Medium Demand|1.0  |4002 |
|High Demand  |2.0  |1674 |
+-------------+-----+-----+

Distribusi label pada data test:
+-------------+-----+-----+
|demand_level |label|count|
+-------------+-----+-----+
|Low Demand   |0.0  |1974 |
|Medium Demand|1.0  |1025 |
|High Demand  |2.0  |427  |
+-------------+-----+-----+



## Menyiapkan Pipeline Machine Learning

Fitur kategorikal diproses menggunakan `StringIndexer` dan `OneHotEncoder`, sedangkan seluruh fitur digabungkan menggunakan `VectorAssembler`.

In [11]:
from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [12]:
indexers = [
    StringIndexer(
        inputCol=feature,
        outputCol=f"{feature}_index",
        handleInvalid="keep"
    )
    for feature in categorical_features
]

encoders = [
    OneHotEncoder(
        inputCol=f"{feature}_index",
        outputCol=f"{feature}_encoded"
    )
    for feature in categorical_features
]

encoded_features = [f"{feature}_encoded" for feature in categorical_features]

assembler = VectorAssembler(
    inputCols=numeric_features + encoded_features,
    outputCol="features",
    handleInvalid="keep"
)

preprocessing_stages = indexers + encoders + [assembler]

## Menyiapkan Evaluator

Metrik evaluasi yang digunakan adalah accuracy, weighted precision, weighted recall, dan F1-score.

In [13]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

In [14]:
def evaluate_model(model_name, fitted_pipeline, test_data):
    predictions = fitted_pipeline.transform(test_data)
    
    result = {
        "model": model_name,
        "accuracy": accuracy_evaluator.evaluate(predictions),
        "precision": precision_evaluator.evaluate(predictions),
        "recall": recall_evaluator.evaluate(predictions),
        "f1_score": f1_evaluator.evaluate(predictions)
    }
    
    return result, predictions

## Model 1 — Logistic Regression

In [15]:
logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50
)

lr_pipeline = Pipeline(
    stages=preprocessing_stages + [logistic_regression]
)

lr_model = lr_pipeline.fit(train_df)

lr_result, lr_predictions = evaluate_model(
    model_name="Logistic Regression",
    fitted_pipeline=lr_model,
    test_data=test_df
)

lr_result

{'model': 'Logistic Regression',
 'accuracy': 0.7530647985989493,
 'precision': 0.7430649105468314,
 'recall': 0.7530647985989491,
 'f1_score': 0.7420179008088271}

## Model 2 — Decision Tree

In [16]:
decision_tree = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=10,
    seed=42
)

dt_pipeline = Pipeline(
    stages=preprocessing_stages + [decision_tree]
)

dt_model = dt_pipeline.fit(train_df)

dt_result, dt_predictions = evaluate_model(
    model_name="Decision Tree",
    fitted_pipeline=dt_model,
    test_data=test_df
)

dt_result

{'model': 'Decision Tree',
 'accuracy': 0.8928779918272037,
 'precision': 0.8913725961143953,
 'recall': 0.8928779918272037,
 'f1_score': 0.8900506486959562}

## Model 3 — Random Forest

In [17]:
random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_pipeline = Pipeline(
    stages=preprocessing_stages + [random_forest]
)

rf_model = rf_pipeline.fit(train_df)

rf_result, rf_predictions = evaluate_model(
    model_name="Random Forest",
    fitted_pipeline=rf_model,
    test_data=test_df
)

rf_result

{'model': 'Random Forest',
 'accuracy': 0.7028604786923526,
 'precision': 0.7315529812482233,
 'recall': 0.7028604786923525,
 'f1_score': 0.6503589661979102}

## Perbandingan Performa Model

In [18]:
evaluation_results = pd.DataFrame([
    lr_result,
    dt_result,
    rf_result
])

evaluation_results = (
    evaluation_results
    .sort_values(by="f1_score", ascending=False)
    .reset_index(drop=True)
)

evaluation_results

,model,accuracy,precision,recall,f1_score
0,Decision Tree,0.892878,0.891373,0.892878,0.890051
1,Logistic Regression,0.753065,0.743065,0.753065,0.742018
2,Random Forest,0.702860,0.731553,0.702860,0.650359


In [19]:
evaluation_result_path = MODELING_DIR / "model_evaluation_results.csv"

evaluation_results.to_csv(evaluation_result_path, index=False)

print("Hasil evaluasi model berhasil disimpan ke:")
print(evaluation_result_path)

Hasil evaluasi model berhasil disimpan ke:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\model_evaluation_results.csv


## Menentukan Model Terbaik

In [20]:
best_model_name = evaluation_results.iloc[0]["model"]

print("Model terbaik berdasarkan F1-score:", best_model_name)

if best_model_name == "Logistic Regression":
    best_model = lr_model
    best_predictions = lr_predictions
elif best_model_name == "Decision Tree":
    best_model = dt_model
    best_predictions = dt_predictions
else:
    best_model = rf_model
    best_predictions = rf_predictions

Model terbaik berdasarkan F1-score: Decision Tree


In [21]:
best_predictions.select(
    "demand_level",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

+-------------+-----+----------+-------------------------------------------------------------+
|demand_level |label|prediction|probability                                                  |
+-------------+-----+----------+-------------------------------------------------------------+
|Low Demand   |0.0  |0.0       |[0.9545454545454546,0.045454545454545456,0.0]                |
|Low Demand   |0.0  |0.0       |[0.9545454545454546,0.045454545454545456,0.0]                |
|Low Demand   |0.0  |0.0       |[0.9545454545454546,0.045454545454545456,0.0]                |
|Low Demand   |0.0  |0.0       |[1.0,0.0,0.0]                                                |
|Medium Demand|1.0  |0.0       |[0.9545454545454546,0.045454545454545456,0.0]                |
|Low Demand   |0.0  |0.0       |[0.9743589743589743,0.02442002442002442,0.001221001221001221]|
|Low Demand   |0.0  |0.0       |[0.9545454545454546,0.045454545454545456,0.0]                |
|Low Demand   |0.0  |0.0       |[0.954545454545454

## Confusion Matrix

In [22]:
prediction_pd = (
    best_predictions
    .select("label", "prediction")
    .toPandas()
)

confusion_matrix = pd.crosstab(
    prediction_pd["label"],
    prediction_pd["prediction"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

confusion_matrix

Predicted,0.0,1.0,2.0
Actual,,,
0.0,1922,36,16
1.0,133,842,50
2.0,79,53,295


In [23]:
label_name_map = {
    0.0: "Low Demand",
    1.0: "Medium Demand",
    2.0: "High Demand"
}

prediction_pd["actual_class"] = prediction_pd["label"].map(label_name_map)
prediction_pd["predicted_class"] = prediction_pd["prediction"].map(label_name_map)

confusion_matrix_named = pd.crosstab(
    prediction_pd["actual_class"],
    prediction_pd["predicted_class"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

confusion_matrix_named

Predicted,High Demand,Low Demand,Medium Demand
Actual,,,
High Demand,295,79,53
Low Demand,16,1922,36
Medium Demand,50,133,842


In [24]:
confusion_matrix_path = MODELING_DIR / "confusion_matrix_best_model.csv"
confusion_matrix_named_path = MODELING_DIR / "confusion_matrix_best_model_named.csv"

confusion_matrix.to_csv(confusion_matrix_path)
confusion_matrix_named.to_csv(confusion_matrix_named_path)

print("Confusion matrix berhasil disimpan ke:")
print(confusion_matrix_path)
print(confusion_matrix_named_path)

Confusion matrix berhasil disimpan ke:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\confusion_matrix_best_model.csv
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\confusion_matrix_best_model_named.csv


## Feature Importance

In [25]:
if best_model_name in ["Decision Tree", "Random Forest"]:
    final_model = best_model.stages[-1]
    feature_importance = final_model.featureImportances.toArray()
    
    feature_importance_df = pd.DataFrame({
        "feature_index": range(len(feature_importance)),
        "importance": feature_importance
    })
    
    feature_importance_df = (
        feature_importance_df
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    
    display(feature_importance_df.head(20))
else:
    print("Feature importance tidak ditampilkan karena model terbaik bukan tree-based model.")

,feature_index,importance
0,5,0.386457
1,0,0.239868
2,2,0.074763
3,13,0.055674
4,14,0.041328
5,3,0.038713
6,15,0.036237
7,21,0.019527
8,17,0.018817
9,16,0.016332


In [26]:
if best_model_name in ["Decision Tree", "Random Forest"]:
    feature_importance_path = MODELING_DIR / "feature_importance_best_model.csv"
    feature_importance_df.to_csv(feature_importance_path, index=False)
    
    print("Feature importance berhasil disimpan ke:")
    print(feature_importance_path)
else:
    print("Feature importance tidak disimpan karena model terbaik bukan tree-based model.")

Feature importance berhasil disimpan ke:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\feature_importance_best_model.csv


## Ringkasan Hasil Model

In [27]:
best_accuracy = evaluation_results.iloc[0]["accuracy"]
best_precision = evaluation_results.iloc[0]["precision"]
best_recall = evaluation_results.iloc[0]["recall"]
best_f1 = evaluation_results.iloc[0]["f1_score"]

print("RINGKASAN HASIL MODEL")
print("=" * 70)
print(f"Model terbaik : {best_model_name}")
print(f"Accuracy      : {best_accuracy:.4f}")
print(f"Precision     : {best_precision:.4f}")
print(f"Recall        : {best_recall:.4f}")
print(f"F1-score      : {best_f1:.4f}")
print("=" * 70)

RINGKASAN HASIL MODEL
Model terbaik : Decision Tree
Accuracy      : 0.8929
Precision     : 0.8914
Recall        : 0.8929
F1-score      : 0.8901


## Ringkasan Akhir Notebook

In [29]:
print("RINGKASAN NOTEBOOK 4")
print("=" * 70)
print(f"Dataset input              : {processed_csv_path}")
print(f"Jumlah data train          : {train_df.count()}")
print(f"Jumlah data test           : {test_df.count()}")
print(f"Jumlah fitur numerik       : {len(numeric_features)}")
print(f"Jumlah fitur kategorikal   : {len(categorical_features)}")
print(f"Model yang dibandingkan    : Logistic Regression, Decision Tree, Random Forest")
print(f"Model terbaik              : {best_model_name}")
print(f"File evaluasi model        : {evaluation_result_path}")
print(f"File confusion matrix      : {confusion_matrix_named_path}")
print("=" * 70)

RINGKASAN NOTEBOOK 4
Dataset input              : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\processed\ecommerce_demand_processed.csv
Jumlah data train          : 13990
Jumlah data test           : 3426
Jumlah fitur numerik       : 13
Jumlah fitur kategorikal   : 5
Model yang dibandingkan    : Logistic Regression, Decision Tree, Random Forest
Model terbaik              : Decision Tree
File evaluasi model        : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\model_evaluation_results.csv
File confusion matrix      : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\results\modeling\confusion_matrix_best_model_named.csv
